# Step B frame disagreement review

Regenerate — **do not hand-edit** the `.ipynb` JSON:

```bash
uv run python scripts/generate_notebooks.py --name frame_disagreement_review
```

**Purpose:** paper / methods prep — skim **COOKING↔CURE** and **CURE↔INGESTION** (and service/use verb rows) where hand gold and `gold_predictions` disagree. Plain **pandas** (works in Cursor).

**Not** a re-labelling session unless you change [ANNOTATION_STRATEGY §2.1](../docs/ANNOTATION_STRATEGY.md). Policy: [GOLD_LABELLING.md § Review disagreements](../docs/GOLD_LABELLING.md#review-disagreements-csv--edit-this-not-excel).

CLI export:

```bash
uv run python scripts/eval_disagreements.py
```


## 0 — Paths & helpers


In [ ]:
from __future__ import annotations

import re
import subprocess
from pathlib import Path

import pandas as pd
from data_io import resolve

pd.set_option('display.max_colwidth', 120)
pd.set_option('display.max_rows', 80)

EVAL = Path(resolve('eval_reports'))
SCRATCH = Path(resolve('trifecta_gold')).parent
FRAME_CSV = EVAL / 'frame_disagreements.csv'
FIXES_CSV = EVAL / 'gold_fixes.csv'
GOLD_ALL = SCRATCH / 'gold_labelling_all.csv'
GOLD_CSV = Path(resolve('trifecta_gold_csv'))

DISPLAY_COLS = [
    'record_id', 'target_word', 'gold_frame', 'pred_frame',
    'gold_lexical_unit', 'pred_lexical_unit', 'pair',
    'text_regime', 'notes', 'context_snippet',
]

BOUNDARY_PAIRS = {
    'COOKING_CURE': {
        ('COOKING_CREATION', 'CURE'),
        ('CURE', 'COOKING_CREATION'),
    },
    'CURE_INGESTION': {
        ('CURE', 'INGESTION'),
        ('INGESTION', 'CURE'),
    },
    'COOKING_INGESTION': {
        ('COOKING_CREATION', 'INGESTION'),
        ('INGESTION', 'COOKING_CREATION'),
    },
}

SERVICE_USE_RE = re.compile(
    r'\b(aanbied|server|gebruik|bedien|toepass)\w*',
    re.I,
)


def pair_key(gold: str, pred: str) -> str:
    a, b = sorted([str(gold or ''), str(pred or '')])
    return f'{a}↔{b}'


def mask_pair_set(frame: pd.DataFrame, pairs: set[tuple[str, str]]) -> pd.Series:
    return frame.apply(
        lambda r: (str(r['gold_frame']), str(r['pred_frame'])) in pairs,
        axis=1,
    )


def walk_rows(frame: pd.DataFrame, start: int = 0, n: int = 5) -> None:
    """Print GOLD vs PRED + notes for paper quotes."""
    chunk = frame.iloc[start : start + n]
    if chunk.empty:
        print('(empty)')
        return
    for i, row in chunk.iterrows():
        print('=' * 72)
        print(f"[{i}] {row.get('record_id')}")
        print(f"  target: {row.get('target_word')!r}  regime: {row.get('text_regime')!r}")
        print(f"  gold={row.get('gold_frame')}  pred={row.get('pred_frame')}  "
              f"pair={row.get('pair')}")
        print(f"  LU gold={row.get('gold_lexical_unit')!r}  "
              f"pred={row.get('pred_lexical_unit')!r}")
        notes = str(row.get('notes') or '').strip()
        if notes:
            print(f"  notes: {notes}")
        print(f"  context: {row.get('context_snippet')}")
    print('=' * 72)


print('EVAL', EVAL)
print('FRAME_CSV exists', FRAME_CSV.exists())


## 1 — Refresh disagreement CSVs (optional)

Needs current `gold_predictions`. Skip if CSVs are already fresh.


In [ ]:
REFRESH = True  # set False to skip re-export

if REFRESH:
    subprocess.run(
        ['uv', 'run', 'python', 'scripts/eval_disagreements.py'],
        check=True,
    )
else:
    print('skipped export')

assert FRAME_CSV.is_file(), f'Missing {FRAME_CSV} — run eval_disagreements.py'


## 2 — Load frame disagreements + gold notes


In [ ]:
frames = pd.read_csv(FRAME_CSV, dtype=str, keep_default_na=False)
frames['pair'] = [
    pair_key(g, p) for g, p in zip(frames['gold_frame'], frames['pred_frame'])
]

gold_path = GOLD_ALL if GOLD_ALL.is_file() else GOLD_CSV
gold = pd.read_csv(gold_path, dtype=str, keep_default_na=False)
note_cols = [c for c in ('record_id', 'notes', 'text_regime', 'selected_frame',
                          'lexical_unit', 'context_text', 'labelled')
             if c in gold.columns]
gold_slim = gold[note_cols].drop_duplicates(subset=['record_id'], keep='last')

df = frames.merge(gold_slim, on='record_id', how='left', suffixes=('', '_gold'))
if 'text_regime' not in df.columns and 'text_regime_gold' in df.columns:
    df['text_regime'] = df['text_regime_gold']
if 'notes' not in df.columns:
    df['notes'] = ''

print(f'frame disagreements: {len(df)}  (gold source: {gold_path.name})')
print('by pair:')
display(df['pair'].value_counts().to_frame('n'))
print('by regime (top):')
display(df['text_regime'].fillna('').value_counts().head(10).to_frame('n'))


## 3 — Confusion / boundary slices

Paper-relevant pairs from [ANNOTATION_STRATEGY §2.1 / §2.4](../docs/ANNOTATION_STRATEGY.md).


In [ ]:
xtab = pd.crosstab(df['gold_frame'], df['pred_frame'], margins=True)
display(xtab)

slices = {
    name: df[mask_pair_set(df, pairs)].copy()
    for name, pairs in BOUNDARY_PAIRS.items()
}
summary = pd.DataFrame(
    [{'slice': k, 'n': len(v), 'with_notes': (v['notes'].astype(str).str.strip() != '').sum()}
     for k, v in slices.items()]
).set_index('slice')
display(summary)

# Default focus for this session (edit):
FOCUS = 'CURE_INGESTION'  # or COOKING_CURE / COOKING_INGESTION
slice_ = slices[FOCUS]
print(f'FOCUS={FOCUS} n={len(slice_)}')
cols = [c for c in DISPLAY_COLS if c in slice_.columns]
display(slice_[cols].head(20))


## 4 — Service / use verbs (`aanbieden`, `serveren`, `gebruiken`, …)

Heuristic on LU + context — not a new frame. Skim for paper examples.


In [ ]:
lu_gold = df['gold_lexical_unit'].fillna('')
lu_pred = df['pred_lexical_unit'].fillna('')
ctx = df['context_snippet'].fillna('')
service = df[
    lu_gold.str.contains(SERVICE_USE_RE, na=False)
    | lu_pred.str.contains(SERVICE_USE_RE, na=False)
    | ctx.str.contains(SERVICE_USE_RE, na=False)
].copy()
print(f'service/use heuristic hits: {len(service)}')
cols = [c for c in DISPLAY_COLS if c in service.columns]
display(service[cols].head(25))


## 5 — Walk details (2–3 paper quotes)

Prefer rows with non-empty `notes`. Bump `start` for the next page.


In [ ]:
with_notes = slice_[slice_['notes'].astype(str).str.strip() != ''].copy()
pool = with_notes if len(with_notes) else slice_
print(f'walking {len(pool)} rows (notes-first={len(with_notes) > 0})')
walk_rows(pool, start=0, n=5)
# walk_rows(pool, start=5, n=5)


## 6 — Optional: shortlist CSV for quotes / gold_fixes skim


In [ ]:
out = EVAL / f'frame_boundary_{FOCUS.lower()}.csv'
cols = [c for c in DISPLAY_COLS if c in slice_.columns]
slice_[cols].to_csv(out, index=False)
print('wrote', out, 'rows', len(slice_))

if FIXES_CSV.is_file():
    fixes = pd.read_csv(FIXES_CSV, dtype=str, keep_default_na=False)
    pending = fixes[fixes['verdict'].astype(str).str.strip() == '']
    print(f'gold_fixes: {len(fixes)} rows, {len(pending)} without verdict')
    # Edit FIXES_CSV in Cursor if adjudicating; then:
    # uv run python scripts/import_gold_fixes.py --import-after
else:
    print('no gold_fixes.csv yet')


## How to use

1. Set `FOCUS` to the boundary you need for the write-up.
2. `walk_rows` — copy 2–3 snippets + `notes` into the paper methods section.
3. Do **not** re-label the same rows unless §2.1 rules change.
4. Optional adjudication: fill `verdict` in `gold_fixes.csv` → `import_gold_fixes.py` (not disagreement-driven sampling alone).
5. For Step C qualia (not frame pairs): use `step_c_review.ipynb`.
